[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [GeoPandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/geopandas-deep-dive.html)

# The GeoDataFrame &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely

COUNTRIES_URL = ("https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/data/nordic-countries.geojson")
local_copy = Path("../../data/nordic-countries.geojson")
countries = gpd.read_file(local_copy if local_copy.exists() else COUNTRIES_URL)


weather_stations = pd.DataFrame({
    "name": ["Bergen", "Oslo", "Svalbard", "Tromso", "Kirkenes"],
    "latitude": [60.39, 59.91, 78.22, 69.65, 69.73],
    "longitude": [5.32, 10.75, 15.65, 18.96, 30.05],
})


def mainland_norway():
    """Norway's outline south of 74 degrees north, without Svalbard, in UTM zone 33N."""
    parts = countries[countries["ADMIN"] == "Norway"].to_crs(25833).explode()
    return parts[parts.geometry.bounds["maxy"] < 8_200_000].union_all()


def station_network(n=300, seed=2026):
    """A made-up register of weather stations on mainland Norway, one row each."""
    rng = np.random.default_rng(seed)
    land = mainland_norway()
    x_min, y_min, x_max, y_max = land.bounds
    east, north = [], []
    while len(east) < n:
        x, y = rng.uniform(x_min, x_max), rng.uniform(y_min, y_max)
        if land.contains(shapely.Point(x, y)):
            east.append(x)
            north.append(y)
    lonlat = gpd.GeoSeries(gpd.points_from_xy(east, north), crs=25833).to_crs(4326)
    return pd.DataFrame({
        "station": [f"NS{i:03d}" for i in range(1, n + 1)],
        "latitude": lonlat.y.round(4).to_numpy(), "longitude": lonlat.x.round(4).to_numpy(),
        "elevation_m": np.clip(rng.gamma(1.6, 220, n), 2, 1900).round(0).astype(int),
        "owner": rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2]),
        "installed": rng.integers(1950, 2025, n),
    })


DISTRICT_CENTERS = {"South": (8.0, 58.6), "East": (10.8, 60.5), "West": (6.0, 60.8), "Mid": (11.0, 63.6),
                    "North": (14.5, 67.0), "Far North": (19.0, 69.4), "Northeast": (25.5, 70.0),
                    "Svalbard": (16.0, 78.5)}


def forecast_districts():
    """Eight made-up forecast districts that cover Norway, in UTM zone 33N."""
    norway = countries[countries["ADMIN"] == "Norway"].to_crs(25833).geometry.iloc[0]
    centers = gpd.GeoSeries(gpd.points_from_xy(*zip(*DISTRICT_CENTERS.values())), crs=4326).to_crs(25833)
    cells = shapely.voronoi_polygons(shapely.MultiPoint(list(centers)),
                                     extend_to=shapely.box(*norway.buffer(200_000).bounds)).geoms
    names = [next(name for name, center in zip(DISTRICT_CENTERS, centers) if cell.contains(center))
             for cell in cells]
    districts = gpd.GeoDataFrame({"district": names}, geometry=[cell.intersection(norway) for cell in cells],
                                 crs=25833)
    return districts.set_index("district").loc[list(DISTRICT_CENTERS)].reset_index()


network = station_network()
stations = gpd.GeoDataFrame(
    network, geometry=gpd.points_from_xy(network["longitude"], network["latitude"]), crs=4326
)


districts = forecast_districts()
print("geopandas", gpd.__version__, "|", len(stations), "made-up stations,", len(districts),
      "made-up districts,", len(countries), "countries")

geopandas 1.2.0 | 300 made-up stations, 8 made-up districts, 5 countries


The worked examples' table of owners, rebuilt here so that each answer runs on its own. The stations and the districts are made up.

In [2]:
owners = pd.DataFrame({
    "owner": ["met", "road", "airport", "private"],
    "operator": ["weather service", "road authority", "airport operator", "private owner"],
})

**1.** Write the five named stations as WKT strings with an f-string, as the worked examples did, but with latitude and longitude rounded to one decimal place, and make a GeoDataFrame from them. Print its CRS and its geometry as WKT.

In [3]:
text = [f"POINT ({lon:.1f} {lat:.1f})"
        for lon, lat in zip(weather_stations["longitude"], weather_stations["latitude"])]
rounded = gpd.GeoDataFrame({"name": weather_stations["name"]}, geometry=gpd.GeoSeries.from_wkt(text), crs=4326)
print(rounded.crs)
print(rounded.to_wkt()["geometry"].tolist())

EPSG:4326
['POINT (5.3 60.4)', 'POINT (10.8 59.9)', 'POINT (15.7 78.2)', 'POINT (19 69.7)', 'POINT (30.1 69.7)']


`:.1f` writes a number with one decimal place, so the text holds the rounded coordinates before it is parsed.

**2.** Keep the stations higher than 1,000 m, and print the type of the result and how many stations it holds.

In [4]:
high = stations[stations["elevation_m"] > 1000]
print(type(high).__name__, len(high))

GeoDataFrame 8


Filtering rows keeps every column, the geometry included, so the result is still a GeoDataFrame.

**3.** Make a copy of `districts` with a column `center` holding each district's centroid, make it the active geometry, and print each district's name with the x and y of its center in kilometers, rounded to a whole number.

In [5]:
centers = districts.assign(center=districts.centroid).set_geometry("center")
for name, point in zip(centers["district"], centers.geometry):
    print(f"{name:10} x {point.x / 1000:6.0f} km   y {point.y / 1000:6.0f} km")

South      x     80 km   y   6562 km
East       x    255 km   y   6738 km
West       x     57 km   y   6799 km
Mid        x    291 km   y   7042 km
North      x    478 km   y   7395 km
Far North  x    662 km   y   7672 km
Northeast  x    912 km   y   7798 km
Svalbard   x    572 km   y   8751 km


The districts are in UTM zone 33N, so the coordinates are meters east and north, and dividing by 1,000 gives kilometers.

**4.** Merge `owners` onto `stations` so that the result is a GeoDataFrame, and print its type, its CRS and the `station`, `owner` and `operator` of its first three rows.

In [6]:
with_operator = stations.merge(owners, on="owner")
print(type(with_operator).__name__, with_operator.crs)
print(with_operator[["station", "owner", "operator"]].head(3))

GeoDataFrame EPSG:4326
  station owner         operator
0   NS001   met  weather service
1   NS002   met  weather service
2   NS003   met  weather service


With the GeoDataFrame on the left, `merge` returns a GeoDataFrame and keeps the CRS.

**5.** Find Norway in `countries` by its three-letter code, without using its name, and print its `POP_EST`. Print the code columns first, since Natural Earth's codes have surprises.

In [7]:
norway = countries[countries["ADM0_A3"] == "NOR"]
print(norway["POP_EST"].iloc[0])

5347896.0


`ISO_A3` would find nothing, since Natural Earth gives Norway `-99` there, so the answer uses `ADM0_A3`.

---

&#8592; **Back to:** [The GeoDataFrame](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/geopandas-deep-dive/03-the-geodataframe.ipynb)  &nbsp;&middot;&nbsp;  [GeoPandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/geopandas-deep-dive.html)
